## Phase 0 - Imports & Configuration

In [1]:
!pip install stanza

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 31.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 773.7/773.7 kB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 418.7/418.7 kB 23.7 MB/s eta 0:00:00


In [2]:
import os
import json
import re
import time
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import stanza
import joblib
from sklearn.svm import SVC
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score, average_precision_score,
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay, f1_score,
    precision_recall_curve, precision_score,
    recall_score, roc_auc_score, roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
from tqdm.notebook import tqdm

In [3]:
DATASET_PATH  = "/kaggle/input/datasets/venesiaarisaputri/prdect-id-dataset/PRDECT-ID Dataset.csv"
_SCRIPT_DIR   = "/kaggle/working"
MODEL_DIR     = os.path.join(_SCRIPT_DIR, "model_weights")
EVAL_DIR      = os.path.join(_SCRIPT_DIR, "evaluation_metrics")
RESULT_DIR    = os.path.join(_SCRIPT_DIR, "results")
SVM_MODEL_PATH  = os.path.join(MODEL_DIR,  "svm_model.joblib")
VECTORIZER_PATH = os.path.join(MODEL_DIR,  "tfidf_svm_vectorizer.joblib")

for d in (MODEL_DIR, EVAL_DIR, RESULT_DIR):
    os.makedirs(d, exist_ok=True)

LABEL2ID       = {"negatif": 0, "positif": 1}
ID2LABEL       = {0: "negatif", 1: "positif"}

SVM_C      = 1.0
SVM_KERNEL = "linear"

TFIDF_MAX_FEATURES  = None
TFIDF_NGRAM_RANGE   = (1, 2)
TFIDF_MIN_DF        = 5
TFIDF_SUBLINEAR_TF  = True
TFIDF_STRIP_ACCENTS = "unicode"

CONFIDENCE_THRESHOLD = 0.75
SIMILARITY_THRESHOLD = 0.99
TOP_N_SUMMARY        = 10
MIN_COUNT            = 3

NEGATION_WORDS = {"tidak", "bukan", "belum", "jangan", "kurang", "tanpa"}
SPLIT_CONJUNCTIONS = ["tapi", "tetapi", "namun", "meski", "meskipun", "walaupun", "walau", "dan", "serta", "juga", "karena", "soalnya", "sebab", "padahal", "sedangkan"]

REGEX_PATTERNS = [
    (r"(.)\1{2,}",          r"\1\1"),   # "bangeeetttt" -> "bangeett"
    (r"([a-zA-Z])\1\b",     r"\1"),     # "bangett" -> "banget"
    (r"([!?,;.]){2,}",      r"\1"),     # ",,,,," -> ","
    (r"\b(\w+)\s+\1\b",     r"\1"),     # "bagus bagus" -> "bagus"
    (r"\s+([!?,;.])",       r"\1"),     # space before punctuation
    (r"([!?,;.])(?!\s)",    r"\1 "),    # space after punctuation
]

SLANG_DICT = {
    "gak": "tidak", "ga": "tidak", "gk": "tidak", "nggak": "tidak", "ngga": "tidak", "engga": "tidak", "tdk": "tidak", "tak": "tidak",
    "tp": "tapi", "tpi": "tapi", "cuma": "hanya", "cmn": "hanya", "cuman": "hanya",
    "bgt": "banget", "bgd": "banget", "bngt": "banget", "bet": "banget", "bnget": "banget", "sgt": "sangat", "sngat": "sangat", "sangt": "sangat",
    "sy": "saya", "sya": "saya", "gw": "saya", "gue": "saya", "aku": "saya", "ak": "saya", "lo": "kamu", "lu": "kamu", "kmu": "kamu",
    "ud": "sudah", "udh": "sudah", "uda": "sudah", "udah": "sudah", "sdh": "sudah", "blm": "belum", "blum": "belum", "belom": "belum", "blom": "belum",
    "lg": "lagi", "lgi": "lagi", "bs": "bisa", "bsa": "bisa", "skrg": "sekarang", "skr": "sekarang", "skg": "sekarang", "cb": "coba", "cba": "coba",
    "br": "baru", "bru": "baru", "cpt": "cepat", "cepet": "cepat", "cpet": "cepat", "lm": "lama", "lma": "lama", "hrs": "harus", "hrus": "harus",
    "bgs": "bagus", "bgus": "bagus", "mntp": "bagus", "mntap": "bagus", "mantep": "bagus", "mntep": "bagus", "mantap": "bagus",
    "sj": "saja", "aja": "saja", "sja": "saja", "aj": "saja", "doank": "saja", "doang": "saja",
    "bbrp": "beberapa", "bbrapa": "beberapa", "bbrpa": "beberapa", "bebrpa": "beberapa", "brp": "berapa", "brpa": "berapa", "brapa": "berapa",
    "bgini": "seperti ini", "begini": "seperti ini", "sperti": "seperti", "yg": "yang", "yng": "yang", "krn": "karena", "karna": "karena",
    "dgn": "dengan", "dg": "dengan", "dr": "dari", "dri": "dari", "utk": "untuk", "buat": "untuk", "jg": "juga", "sm": "sama", "mk": "maka",
    "brg": "barang", "brng": "barang", "ongkir": "ongkos kirim", "ori": "original", "ok": "oke", "pngiriman": "pengiriman", "pngirim": "pengirim", "krm": "kirim",
    "saller": "penjual", "saler": "penjual", "seler": "penjual", "seller": "penjual", "pnjual": "penjual",
    "lmyn": "lumayan", "lmayan": "lumayan", "kualits": "kualitas", "packing": "packaging",
    "wkwk": "", "wkwkwk": "", "haha": "", "hihi": "", "hehe": "", "masyaallah": "", "sih": "", "alhamdulilah": "", "alhamdullilah": "", "alhamdulillah": "",
    "mantul": "bagus banget", "gokil": "luar biasa", "okesip": "oke siap"
}

SVM_LEXICON: set = set()


In [4]:
stanza.download("id", verbose=False)
nlp_stanza = stanza.Pipeline(
    "id",
    processors="tokenize,mwt,pos,lemma,depparse",
    verbose=False,
)

## Phase 1 - Data Loading & Preprocessing

In [5]:
_CLEAN_WORD_RE = re.compile(r'^[a-zA-Z]{3,}$')

def is_clean_word(word: str) -> bool:
    return bool(_CLEAN_WORD_RE.match(word))

def apply_regex_patterns(text: str) -> str:
    for pattern, replacement in REGEX_PATTERNS:
        text = re.sub(pattern, replacement, text)
    return text

def normalize_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    words = text.split()
    normalized = []
    for word in words:
        replacement = SLANG_DICT.get(word, word)
        if replacement:
            normalized.extend(replacement.split())

    text = " ".join(normalized)
    text = apply_regex_patterns(text)
    text = re.sub(r"[^\w\s,.!?]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def load_and_clean_data(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    df = df.rename(columns={
        "Customer Review": "review_text",
        "Customer Rating": "rating",
    })
    df["review_text"] = df["review_text"].astype(str)
    df["rating"]      = pd.to_numeric(df["rating"], errors="coerce")
    df = df.dropna(subset=["rating"])
    df["rating"] = df["rating"].astype(int)
    df = df[df["review_text"].str.split().str.len() >= 5]
    df = df[df["review_text"].str.contains(r"[a-zA-Z]", regex=True)]
    df = df[df["review_text"].str.contains(r"[a-zA-Z0-9\s.,!?]", regex=True)]
    df = df.drop_duplicates(subset=["review_text"])
    df = df.reset_index(drop=True)
    df["review_normalized"] = df["review_text"].apply(normalize_text)

    def assign_label(sentiment: str):
        if sentiment == "Negative": return 0
        elif sentiment == "Positive": return 1
        return None

    df["label"] = df["Sentiment"].apply(assign_label)
    print(f"Total after cleaning : {len(df)} rows")
    print(f"Rating distribution  :\n{df['rating'].value_counts().sort_index()}")
    print(f"\nLabel distribution:")
    print(f"Positive (1): {(df['label'] == 1).sum()}")
    print(f"Negative (0): {(df['label'] == 0).sum()}")
    return df

df_raw = load_and_clean_data(DATASET_PATH)
df_train_pool = df_raw[df_raw["label"].notna()].copy()
df_train_pool["label"] = df_train_pool["label"].astype(int)

print()
print("Normalization samples:")
for i in range(min(3, len(df_raw))):
    print(f"ORIGINAL   : {df_raw['review_text'].iloc[i]}")
    print(f"NORMALIZED : {df_raw['review_normalized'].iloc[i]}")
    print()

Total after cleaning : 4638 rows
Rating distribution  :
rating
1    1578
2     501
3     412
4     307
5    1840
Name: count, dtype: int64

Label distribution:
Positive (1): 2183
Negative (0): 2455

Normalization samples:
ORIGINAL   : Alhamdulillah berfungsi dengan baik. Packaging aman. Respon cepat dan ramah. Seller dan kurir amanah
NORMALIZED : berfungsi dengan baik. packaging aman. respon cepat dan ramah. penjual dan kurir amanah

ORIGINAL   : barang bagus dan respon cepat, harga bersaing dengan yg lain.
NORMALIZED : barang bagus dan respon cepat, harga bersaing dengan yang lain.

ORIGINAL   : barang bagus, berfungsi dengan baik, seler ramah, pengiriman cepat
NORMALIZED : barang bagus, berfungsi dengan baik, penjual ramah, pengiriman cepat



## Phase 2 - SVM Model Training

In [6]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(df_train_pool['review_normalized'].values, df_train_pool['label'].values, test_size=0.10, random_state=42, stratify=df_train_pool['label'].values)

print(f"Stratified Splitting Complete!")
print(f"Total training samples: {len(X_train_raw)}")
print(f"Total testing samples:  {len(X_test_raw)}")
print(f"Class distribution in Training Set: {Counter(y_train)}")

texts  = df_train_pool["review_normalized"].tolist()
labels = df_train_pool["label"].tolist()
X_train, X_val, y_train, y_val = train_test_split(texts, labels, test_size=0.1, random_state=42, stratify=labels)
print(f"Train samples: {len(X_train)} | Val samples: {len(X_val)}")

if os.path.exists(SVM_MODEL_PATH) and os.path.exists(VECTORIZER_PATH):
    print("Loading saved SVM model and Vectorizer - skipping training.")
    svm_model  = joblib.load(SVM_MODEL_PATH)
    vectorizer = joblib.load(VECTORIZER_PATH)
    X_val_vec  = vectorizer.transform(X_val)
    val_preds  = svm_model.predict(X_val_vec)
    val_acc    = accuracy_score(y_val, val_preds)
    val_f1     = f1_score(y_val, val_preds, average="weighted")
    print(f"Val Accuracy: {val_acc:.4f} | Val F1: {val_f1:.4f}")

else:
    print("Initializing TF-IDF Vectorizer & SVM Classifier...")
    vectorizer = TfidfVectorizer(
        max_features=TFIDF_MAX_FEATURES,
        ngram_range=TFIDF_NGRAM_RANGE,
        min_df=TFIDF_MIN_DF,
        sublinear_tf=TFIDF_SUBLINEAR_TF,
        strip_accents=TFIDF_STRIP_ACCENTS,
    )
    X_train_vec = vectorizer.fit_transform(X_train)
    X_val_vec   = vectorizer.transform(X_val)

    svm_model = SVC(C=SVM_C, kernel=SVM_KERNEL, probability=True, random_state=42)
    print("Fitting SVM model to training data space...")
    import time as _time
    start_time = _time.time()
    svm_model.fit(X_train_vec, y_train)
    print(f"Training finalized in {_time.time() - start_time:.2f} seconds.")

    train_preds = svm_model.predict(X_train_vec)
    train_acc   = accuracy_score(y_train, train_preds)
    train_f1    = f1_score(y_train, train_preds, average="weighted")
    train_prec  = precision_score(y_train, train_preds, average="weighted")
    train_rec   = recall_score(y_train, train_preds, average="weighted")
    val_preds   = svm_model.predict(X_val_vec)
    val_acc     = accuracy_score(y_val, val_preds)
    val_f1      = f1_score(y_val, val_preds, average="weighted")
    val_prec    = precision_score(y_val, val_preds, average="weighted")
    val_rec     = recall_score(y_val, val_preds, average="weighted")

    print(f"Train Accuracy: {train_acc:.4f} | Train F1: {train_f1:.4f}")
    print(f"Val   Accuracy: {val_acc:.4f} | Val   F1: {val_f1:.4f}")

    joblib.dump(svm_model, SVM_MODEL_PATH)
    joblib.dump(vectorizer, VECTORIZER_PATH)
    print(f"SVM Model saved to: '{SVM_MODEL_PATH}'")
    print(f"Vectorizer saved to: '{VECTORIZER_PATH}'")

Stratified Splitting Complete!
Total training samples: 4174
Total testing samples:  464
Class distribution in Training Set: Counter({np.int64(0): 2209, np.int64(1): 1965})
Train samples: 4174 | Val samples: 464
Initializing TF-IDF Vectorizer & SVM Classifier...
Fitting SVM model to training data space...
Training finalized in 6.21 seconds.
Train Accuracy: 0.9880 | Train F1: 0.9880
Val   Accuracy: 0.9698 | Val   F1: 0.9698
SVM Model saved to: '/kaggle/working/model_weights/svm_model.joblib'
Vectorizer saved to: '/kaggle/working/model_weights/tfidf_svm_vectorizer.joblib'


In [7]:
X_train_vec_full = vectorizer.transform(X_train)
train_preds_eval = svm_model.predict(X_train_vec_full)
train_probs_eval = svm_model.predict_proba(X_train_vec_full)[:, 1]
X_test_vec = vectorizer.transform(X_test_raw)
y_pred     = svm_model.predict(X_test_vec)
y_probs    = svm_model.predict_proba(X_test_vec)[:, 1]
tr_acc   = accuracy_score(y_train, train_preds_eval)
tr_f1    = f1_score(y_train, train_preds_eval, average="weighted")
tr_prec  = precision_score(y_train, train_preds_eval, average="weighted")
tr_rec   = recall_score(y_train, train_preds_eval, average="weighted")
tr_cm    = confusion_matrix(y_train, train_preds_eval)
tr_tn, tr_fp, tr_fn, tr_tp = tr_cm.ravel()
tr_spec  = tr_tn / (tr_tn + tr_fp)
tr_auc_roc = roc_auc_score(y_train, train_probs_eval)
_, _, _ = precision_recall_curve(y_train, train_probs_eval)
tr_auc_pr = average_precision_score(y_train, train_probs_eval)
tr_clf_report = classification_report(y_train, train_preds_eval, target_names=["negatif", "positif"], output_dict=True, digits=4)

X_val_vec_eval = vectorizer.transform(X_val)
val_preds_eval = svm_model.predict(X_val_vec_eval)
val_probs_eval = svm_model.predict_proba(X_val_vec_eval)[:, 1]
val_acc  = accuracy_score(y_val, val_preds_eval)
val_f1   = f1_score(y_val, val_preds_eval, average="weighted")
val_prec = precision_score(y_val, val_preds_eval, average="weighted")
val_rec  = recall_score(y_val, val_preds_eval, average="weighted")
val_cm   = confusion_matrix(y_val, val_preds_eval)
val_tn, val_fp, val_fn, val_tp = val_cm.ravel()
val_spec = val_tn / (val_tn + val_fp)
val_auc_roc = roc_auc_score(y_val, val_probs_eval)
val_auc_pr  = average_precision_score(y_val, val_probs_eval)
val_clf_report = classification_report(y_val, val_preds_eval, target_names=["negatif", "positif"], output_dict=True, digits=4)

print("=" * 55)
print("         MODEL EVALUATION RESULTS (Train vs Val)")
print("=" * 55)
print(f"{'Metric':<25} {'Train':>10} {'Val':>10}")
print("-" * 45)
print(f"{'Accuracy':<25} {tr_acc:>10.4f} {val_acc:>10.4f}")
print(f"{'F1 (weighted)':<25} {tr_f1:>10.4f} {val_f1:>10.4f}")
print(f"{'Precision (weighted)':<25} {tr_prec:>10.4f} {val_prec:>10.4f}")
print(f"{'Recall (weighted)':<25} {tr_rec:>10.4f} {val_rec:>10.4f}")
print(f"{'Specificity':<25} {tr_spec:>10.4f} {val_spec:>10.4f}")
print(f"{'AUC-ROC':<25} {tr_auc_roc:>10.4f} {val_auc_roc:>10.4f}")
print(f"{'AUC-PR':<25} {tr_auc_pr:>10.4f} {val_auc_pr:>10.4f}")

fig1, ax1 = plt.subplots(figsize=(6, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=val_cm, display_labels=["negatif", "positif"])
disp.plot(ax=ax1, colorbar=False, cmap="Blues")
ax1.set_title("Confusion Matrix (Validation)", fontsize=13, fontweight="bold")
fig1.tight_layout()
cm_path = os.path.join(EVAL_DIR, "svm_confusion_matrix.png")
fig1.savefig(cm_path, dpi=150, bbox_inches="tight")
plt.close(fig1)
print(f"Confusion matrix saved to {cm_path}")

fpr, tpr, _ = roc_curve(y_val, val_probs_eval)
prec_curve, rec_curve, _ = precision_recall_curve(y_val, val_probs_eval)

fig3, (ax3, ax4) = plt.subplots(1, 2, figsize=(12, 5))
fig3.suptitle("ROC Curve & Precision-Recall Curve - SVM", fontsize=13, fontweight="bold")
ax3.plot(fpr, tpr, color="#7ea5f8", linewidth=2, label=f"AUC-ROC = {val_auc_roc:.4f}")
ax3.plot([0, 1], [0, 1], "k--", linewidth=1)
ax3.set_xlabel("False Positive Rate")
ax3.set_ylabel("True Positive Rate")
ax3.set_title("ROC Curve")
ax3.legend(loc="lower right")
ax3.grid(True, alpha=0.3)
ax4.plot(rec_curve, prec_curve, color="#73e39c", linewidth=2, label=f"AUC-PR = {val_auc_pr:.4f}")
ax4.set_xlabel("Recall")
ax4.set_ylabel("Precision")
ax4.set_title("Precision-Recall Curve")
ax4.legend(loc="lower left")
ax4.grid(True, alpha=0.3)
fig3.tight_layout()
roc_pr_path = os.path.join(EVAL_DIR, "svm_roc_pr_curves.png")
fig3.savefig(roc_pr_path, dpi=150, bbox_inches="tight")
plt.close(fig3)
print(f"ROC & PR curves saved to {roc_pr_path}")

metrics_labels = ["Accuracy", "F1", "Precision", "Recall"]
train_vals_tv  = [tr_acc, tr_f1, tr_prec, tr_rec]
val_vals_tv    = [val_acc, val_f1, val_prec, val_rec]

x = np.arange(len(metrics_labels))
width = 0.35
fig2, ax2 = plt.subplots(figsize=(8, 5))
bars1 = ax2.bar(x - width/2, train_vals_tv, width, label="Train",      color="#7ea5f8")
bars2 = ax2.bar(x + width/2, val_vals_tv,   width, label="Validation", color="#73e39c")
ax2.set_ylim(0, 1.1)
ax2.set_ylabel("Score")
ax2.set_title("Train vs Validation Metrics - SVM", fontsize=13, fontweight="bold")
ax2.set_xticks(x)
ax2.set_xticklabels(metrics_labels)
ax2.legend()
ax2.bar_label(bars1, fmt="%.4f", padding=3, fontsize=8)
ax2.bar_label(bars2, fmt="%.4f", padding=3, fontsize=8)
ax2.grid(True, axis="y", alpha=0.3)
fig2.tight_layout()
tv_path = os.path.join(EVAL_DIR, "svm_train_vs_val_metrics.png")
fig2.savefig(tv_path, dpi=150, bbox_inches="tight")
plt.close(fig2)
print(f"Train vs Val metrics saved -> {tv_path}")

te_acc     = accuracy_score(y_test, y_pred)
te_f1      = f1_score(y_test, y_pred, average="weighted")
te_prec    = precision_score(y_test, y_pred, average="weighted")
te_rec     = recall_score(y_test, y_pred, average="weighted")
te_cm      = confusion_matrix(y_test, y_pred)
te_tn, te_fp, te_fn, te_tp = te_cm.ravel()
te_spec    = te_tn / (te_tn + te_fp)
te_auc_roc = roc_auc_score(y_test, y_probs)
te_auc_pr  = average_precision_score(y_test, y_probs)
te_clf_report = classification_report(y_test, y_pred, target_names=["negatif", "positif"], output_dict=True, digits=4)

all_metrics = {
    "train": {
        "accuracy"             : round(tr_acc,     4),
        "f1_weighted"          : round(tr_f1,      4),
        "precision_weighted"   : round(tr_prec,    4),
        "recall_weighted"      : round(tr_rec,     4),
        "specificity"          : round(tr_spec,    4),
        "auc_roc"              : round(tr_auc_roc, 4),
        "auc_pr"               : round(tr_auc_pr,  4),
        "confusion_matrix"     : tr_cm.tolist(),
        "classification_report": tr_clf_report,
    },
    "validation": {
        "accuracy"             : round(val_acc,     4),
        "f1_weighted"          : round(val_f1,      4),
        "precision_weighted"   : round(val_prec,    4),
        "recall_weighted"      : round(val_rec,     4),
        "specificity"          : round(val_spec,    4),
        "auc_roc"              : round(val_auc_roc, 4),
        "auc_pr"               : round(val_auc_pr,  4),
        "confusion_matrix"     : val_cm.tolist(),
        "classification_report": val_clf_report,
    },
    "test": {
        "accuracy"             : round(te_acc,     4),
        "f1_weighted"          : round(te_f1,      4),
        "precision_weighted"   : round(te_prec,    4),
        "recall_weighted"      : round(te_rec,     4),
        "specificity"          : round(te_spec,    4),
        "auc_roc"              : round(te_auc_roc, 4),
        "auc_pr"               : round(te_auc_pr,  4),
        "confusion_matrix"     : te_cm.tolist(),
        "classification_report": te_clf_report,
    },
}
metrics_json_path = os.path.join(EVAL_DIR, "svm_all_metrics.json")
with open(metrics_json_path, "w", encoding="utf-8") as f:
    json.dump(all_metrics, f, indent=2, ensure_ascii=False)
print(f"All metrics saved to {metrics_json_path}")

         MODEL EVALUATION RESULTS (Train vs Val)
Metric                         Train        Val
---------------------------------------------
Accuracy                      0.9880     0.9698
F1 (weighted)                 0.9880     0.9698
Precision (weighted)          0.9881     0.9706
Recall (weighted)             0.9880     0.9698
Specificity                   0.9955     0.9919
AUC-ROC                       0.9997     0.9903
AUC-PR                        0.9997     0.9915
Confusion matrix saved to /kaggle/working/evaluation_metrics/svm_confusion_matrix.png
ROC & PR curves saved to /kaggle/working/evaluation_metrics/svm_roc_pr_curves.png
Train vs Val metrics saved -> /kaggle/working/evaluation_metrics/svm_train_vs_val_metrics.png
All metrics saved to /kaggle/working/evaluation_metrics/svm_all_metrics.json


## Phase 3 - SVM-based Opinion Lexicon (IG Replacement)

In [8]:
feature_names = np.array(vectorizer.get_feature_names_out())
if hasattr(svm_model.coef_, "toarray"):
    coefficients = svm_model.coef_.toarray()[0]
else:
    coefficients = svm_model.coef_[0]

top_negative_idx = np.argsort(coefficients)[:TOP_N_SUMMARY]
top_positive_idx = np.argsort(coefficients)[-TOP_N_SUMMARY:]
for idx in np.concatenate([top_negative_idx, top_positive_idx]):
    SVM_LEXICON.add(feature_names[idx])
print(f"SVM_LEXICON populated with {len(SVM_LEXICON)} tokens.")

positive_lexicon = {}
negative_lexicon = {}

for word, coef in zip(feature_names, coefficients):
    if not is_clean_word(word):
        continue
    if coef > 0:
        positive_lexicon[word] = float(coef)
    elif coef < 0:
        negative_lexicon[word] = float(coef)

sorted_positive = sorted(positive_lexicon.items(), key=lambda x: x[1], reverse=True)
sorted_negative = sorted(negative_lexicon.items(), key=lambda x: x[1])

print(f"Positive lexicon tokens (filtered): {len(positive_lexicon)}")
print(f"Negative lexicon tokens (filtered): {len(negative_lexicon)}")

lexicon_json_data = {
    "metadata": {
        "total_positive_tokens": len(positive_lexicon),
        "total_negative_tokens": len(negative_lexicon),
        "confidence_threshold" : CONFIDENCE_THRESHOLD,
        "filter_rules"         : "letters only, length > 2",
    },
    "positive_tokens": dict(sorted_positive),
    "negative_tokens": dict(sorted_negative),
}

lexicon_path = os.path.join(RESULT_DIR, "svm_lexicon.json")
with open(lexicon_path, "w", encoding="utf-8") as f:
    json.dump(lexicon_json_data, f, indent=4, ensure_ascii=False)
print(f"SVM lexicon saved to {lexicon_path}")

SVM_LEXICON populated with 20 tokens.
Positive lexicon tokens (filtered): 647
Negative lexicon tokens (filtered): 757
SVM lexicon saved to /kaggle/working/results/svm_lexicon.json


## Phase 4 - Parsing

#### Clause Splitting on Conjunctions and Punctuation

In [9]:
def split_into_clauses(text: str) -> list:
    if not isinstance(text, str) or not text.strip():
        return []
    conj_pattern    = r"\b(?:" + "|".join(map(re.escape, SPLIT_CONJUNCTIONS)) + r")\b"
    processed_text  = re.sub(conj_pattern, "|", text, flags=re.IGNORECASE)
    processed_text  = re.sub(r"[.,!?;\s*]{2,}|[.,!?;]", "|", processed_text)
    raw_clauses     = processed_text.split("|")
    all_clauses     = []
    for clause in raw_clauses:
        cleaned_clause = re.sub(r"\s+", " ", clause).strip()
        if cleaned_clause:
            all_clauses.append(cleaned_clause)
    return all_clauses

sample_input  = "pengiriman cepat tapi produk jelek,, packing rapi dan admin ramah .. ."
sample_output = split_into_clauses(sample_input)
print(f"\nClause splitting example")
print(f"INPUT  : {sample_input}")
print(f"OUTPUT : {sample_output}")


Clause splitting example
INPUT  : pengiriman cepat tapi produk jelek,, packing rapi dan admin ramah .. .
OUTPUT : ['pengiriman cepat', 'produk jelek', 'packing rapi', 'admin ramah']


#### POS & Dependency Parsing + Phrase Extraction (Stanza)

In [10]:
def pos_and_dep_parse(text: str) -> list[dict]:
    if not isinstance(text, str) or not text.strip():
        return []

    doc    = nlp_stanza(text)
    tokens = []
    for sent in doc.sentences:
        for word in sent.words:
            tokens.append({
                "id"     : word.id - 1,
                "word"   : word.lemma.lower() if word.lemma else word.text.lower(),
                "pos"    : word.upos,
                "dep"    : word.deprel,
                "head_id": word.head - 1,
            })
    return tokens

def extract_phrases(clause: str) -> list[dict]:
    tokens = pos_and_dep_parse(clause)
    if not tokens:
        return []

    phrases  = []
    seen     = set()
    n        = len(tokens)
    NOUN_POS = {"NOUN", "PROPN"}
    ADJ_POS  = {"ADJ"}
    ADV_POS  = {"ADV"}
    VERB_POS = {"VERB"}

    def is_neg(tok):
        return tok["word"] in NEGATION_WORDS

    def add(tokens_in_phrase, pattern):
        text = " ".join(t["word"] for t in tokens_in_phrase)
        if text not in seen:
            seen.add(text)
            phrases.append({"phrase": text, "pattern": pattern})

    for i in range(n - 1):
        if tokens[i]["pos"] in NOUN_POS and tokens[i+1]["pos"] in ADJ_POS:
            if not is_neg(tokens[i]) and not is_neg(tokens[i+1]):
                add([tokens[i], tokens[i+1]], "R1_NOUN+ADJ")
    for i in range(n - 2):
        if (tokens[i]["pos"] in NOUN_POS and tokens[i+1]["pos"] in ADV_POS and tokens[i+2]["pos"] in ADJ_POS and not is_neg(tokens[i+1])):
            add([tokens[i], tokens[i+1], tokens[i+2]], "R2_NOUN+ADV+ADJ")
    for i in range(n - 2):
        if (tokens[i]["pos"] in NOUN_POS and is_neg(tokens[i+1]) and tokens[i+2]["pos"] in ADJ_POS):
            add([tokens[i], tokens[i+1], tokens[i+2]], "R3_NOUN+NEG+ADJ")
    for i in range(n - 3):
        if (tokens[i]["pos"] in NOUN_POS and is_neg(tokens[i+1]) and tokens[i+2]["pos"] in ADV_POS and tokens[i+3]["pos"] in ADJ_POS):
            add([tokens[i], tokens[i+1], tokens[i+2], tokens[i+3]], "R4_NOUN+NEG+ADV+ADJ")
    for i in range(n - 1):
        if tokens[i]["pos"] in ADJ_POS and tokens[i+1]["pos"] in NOUN_POS:
            if tokens[i+1]["dep"] in {"root", "nsubj"}:
                add([tokens[i+1], tokens[i]], "R5_ADJ+NOUN")
    for i in range(n - 2):
        if (tokens[i]["pos"] in NOUN_POS and tokens[i+1]["pos"] in VERB_POS and tokens[i+2]["pos"] in ADJ_POS):
            add([tokens[i], tokens[i+1], tokens[i+2]], "R6_NOUN+VERB+ADJ")
    for i in range(n - 3):
        if (tokens[i]["pos"] in NOUN_POS and tokens[i+1]["pos"] in VERB_POS and is_neg(tokens[i+2]) and tokens[i+3]["pos"] in ADJ_POS):
            add([tokens[i], tokens[i+1], tokens[i+2], tokens[i+3]], "R7_NOUN+VERB+NEG+ADJ")
    for i in range(n - 2):
        if (tokens[i]["pos"] in NOUN_POS and tokens[i+1]["pos"] in NOUN_POS and tokens[i+2]["pos"] in ADJ_POS):
            add([tokens[i], tokens[i+1], tokens[i+2]], "R8_NOUN+NOUN+ADJ")
    for i in range(n - 3):
        if (tokens[i]["pos"] in NOUN_POS and tokens[i+1]["pos"] in NOUN_POS and is_neg(tokens[i+2]) and tokens[i+3]["pos"] in ADJ_POS):
            add([tokens[i], tokens[i+1], tokens[i+2], tokens[i+3]], "R9_NOUN+NOUN+NEG+ADJ")
    for i in range(n - 2):
        if (tokens[i]["pos"] in NOUN_POS and tokens[i+1]["pos"] in ADJ_POS and tokens[i+2]["pos"] in ADJ_POS):
            add([tokens[i], tokens[i+1], tokens[i+2]], "R10_NOUN+ADJ+ADJ")
    for token in tokens:
        if token["dep"] == "conj":
            head_id = token["head_id"]
            if 0 <= head_id < n:
                head = tokens[head_id]
                if head["pos"] in NOUN_POS and token["pos"] in ADJ_POS:
                    add([head, token], "R11_conj+shared_head")
                elif head["pos"] in ADJ_POS and token["pos"] in ADJ_POS:
                    for other in tokens:
                        if other["pos"] in NOUN_POS and other["head_id"] == head_id:
                            add([other, head, token], "R11_conj+shared_head")
                            break

    return phrases

test_clause = "pengiriman sangat cepat"
print("\nPOS + Dependency parse:")
for t in pos_and_dep_parse(test_clause):
    print(f"  {t['word']:15} POS={t['pos']:6} DEP={t['dep']}")
print("\nPhrase extraction:")
for p in extract_phrases(test_clause):
    print(f"  {p}")


POS + Dependency parse:
  pengiriman      POS=NOUN   DEP=root
  sangat          POS=ADV    DEP=advmod
  cepat           POS=ADJ    DEP=amod

Phrase extraction:
  {'phrase': 'pengiriman sangat cepat', 'pattern': 'R2_NOUN+ADV+ADJ'}


## Phase 5 - SVM Lexicon Filter

In [11]:
def filter_by_svm_lexicon(phrase_list: list[dict]) -> list[dict]:
    if not SVM_LEXICON:
        print("Warning: SVM_LEXICON is empty. Skipping lexicon filter.")
        return phrase_list
    filtered_phrases = []
    for p in phrase_list:
        words_in_phrase = p["phrase"].lower().split()
        if any(w in SVM_LEXICON for w in words_in_phrase):
            filtered_phrases.append(p)
    return filtered_phrases

## Phase 6 - Phrase Sentiment Prediction

In [12]:
def predict_sentiment_svm_batch(texts: list[str]) -> list[dict]:
    if not texts:
        return []
    vecs  = vectorizer.transform(texts)
    probs = svm_model.predict_proba(vecs)
    results = []
    for prob in probs:
        label_id = int(np.argmax(prob))
        results.append({"label": ID2LABEL[label_id],"score": float(prob[label_id])})
    return results

def predict_phrases_sentiment(phrase_list: list[dict]) -> list[dict]:
    if not phrase_list:
        return []
    texts       = [p["phrase"] for p in phrase_list]
    predictions = predict_sentiment_svm_batch(texts)
    return [
        {
            "phrase"   : pd_item["phrase"],
            "sentiment": pred["label"],
            "score"    : pred["score"],
            "pattern"  : pd_item["pattern"],
        }
        for pd_item, pred in zip(phrase_list, predictions)
    ]

## Phase 7 - Semantic Deduplication (TF-IDF cosine similarity)

In [13]:
def get_phrase_embeddings_svm(phrases: list[str]) -> np.ndarray:
    if not phrases:
        return np.empty((0, 0))
    vecs = vectorizer.transform(phrases)
    return vecs.toarray()

def semantic_deduplication(phrase_sentiment_list: list[dict], threshold: float = SIMILARITY_THRESHOLD) -> list[dict]:
    if not phrase_sentiment_list:
        return []

    def cluster_phrases(phrases):
        if not phrases:
            return []
        texts          = [p["phrase"] for p in phrases]
        freq           = Counter(texts)
        unique_phrases = list(freq.keys())
        unique_counts  = [freq[p] for p in unique_phrases]

        if len(unique_phrases) <= 1:
            return [{"phrase": unique_phrases[0], "count": unique_counts[0]}] if unique_phrases else []

        embeddings = get_phrase_embeddings_svm(unique_phrases)
        sim_matrix = cosine_similarity(embeddings)
        merged     = [False] * len(unique_phrases)
        result     = []

        for i in range(len(unique_phrases)):
            if merged[i]:
                continue
            cluster_count = unique_counts[i]
            for j in range(i + 1, len(unique_phrases)):
                if not merged[j] and sim_matrix[i][j] >= threshold:
                    cluster_count += unique_counts[j]
                    merged[j]      = True
            result.append({"phrase": unique_phrases[i], "count": cluster_count})
        return result

    print("Running semantic deduplication using SVM feature similarity weights...")
    pos_clustered = cluster_phrases([p for p in phrase_sentiment_list if p["sentiment"] == "positif"])
    neg_clustered = cluster_phrases([p for p in phrase_sentiment_list if p["sentiment"] == "negatif"])
    final = (
        [{**item, "sentiment": "positif"} for item in pos_clustered] +
        [{**item, "sentiment": "negatif"} for item in neg_clustered]
    )
    return final

## Phase 8 - Full Pipeline

In [14]:
def run_full_pipeline(df: pd.DataFrame, top_n: int = TOP_N_SUMMARY, min_count: int = MIN_COUNT):
    all_phrases = []
    total       = len(df)
    start       = time.time()

    print(f"Processing {total} reviews...\n")
    with tqdm(total=total, desc="Phrase Extraction", unit="review") as pbar:
        for i, (_, row) in enumerate(df.iterrows()):
            review  = row["review_normalized"]
            clauses = split_into_clauses(review) or [review]
            for clause in clauses:
                all_phrases.extend(extract_phrases(clause))
            pbar.update(1)
            if i % 10 == 0:
                elapsed = time.time() - start
                rate    = (i + 1) / elapsed if elapsed > 0 else 0
                eta     = (total - i) / rate if rate > 0 else 0
                pbar.set_postfix({
                    "phrases": len(all_phrases),
                    "rate"   : f"{rate:.1f} rev/s",
                    "ETA"    : f"{int(eta//60)}m {int(eta%60):02d}s",
                })

    print()
    print(f"Extraction complete in {int((time.time()-start)//60)}m {int((time.time()-start)%60):02d}s")
    print(f"Total phrases extracted: {len(all_phrases)}")

    if not all_phrases:
        print("No phrases extracted.")
        return None, None, {"total_extracted": 0, "total_accepted": 0, "total_rejected": 0}

    all_phrases = filter_by_svm_lexicon(all_phrases)
    print(f"Phrases after SVM Lexicon filter: {len(all_phrases)}")

    print()
    print("Predicting phrase sentiment...")
    accepted = []
    with tqdm(total=len(all_phrases), desc="Sentiment Prediction", unit="phrase") as pbar2:
        for i in range(0, len(all_phrases), 64):
            batch  = all_phrases[i : i + 64]
            result = predict_phrases_sentiment(batch)
            for res in result:
                if res["score"] >= CONFIDENCE_THRESHOLD:
                    accepted.append(res)
            pbar2.update(len(batch))
            pbar2.set_postfix({
                "accepted": len(accepted),
                "rejected": (i + len(batch)) - len(accepted),
            })

    total_accepted = len(accepted)
    total_rejected = len(all_phrases) - total_accepted

    print()
    print(f"Phrases accepted: {total_accepted}")
    print(f"Phrases rejected: {total_rejected} (confidence < {CONFIDENCE_THRESHOLD})")

    phrase_stats = {
        "total_extracted": len(all_phrases),
        "total_accepted" : total_accepted,
        "total_rejected" : total_rejected,
    }

    deduplicated = semantic_deduplication(accepted)

    pos_raw = sorted([p for p in deduplicated if p["sentiment"] == "positif"], key=lambda x: x["count"], reverse=True)
    neg_raw = sorted([p for p in deduplicated if p["sentiment"] == "negatif"], key=lambda x: x["count"], reverse=True)

    print(f"Unique positive clusters: {len(pos_raw)}")
    print(f"Unique negative clusters: {len(neg_raw)}")

    for p in pos_raw + neg_raw:
        p["pct"] = round(min(p["count"] / total * 100, 100.0), 1)

    pos_phrases = [p for p in pos_raw[:top_n] if p["count"] >= min_count]
    neg_phrases = [p for p in neg_raw[:top_n] if p["count"] >= min_count]

    elapsed_total = time.time() - start
    print()
    print(f"Pipeline complete in {int(elapsed_total//60)}m {int(elapsed_total%60):02d}s")

    return pos_phrases, neg_phrases, phrase_stats

df_sample = df_raw.reset_index(drop=True)
pos_summary, neg_summary, phrase_stats = run_full_pipeline(df_sample, top_n=TOP_N_SUMMARY, min_count=MIN_COUNT)

if pos_summary is not None and neg_summary is not None:
    print("\n" + "=" * 55)
    print("          AUTOMATED REVIEW SUMMARY")
    print("=" * 55)
    print("\nPOSITIVE SUMMARY")
    print("-" * 55)
    for i, p in enumerate(pos_summary, 1):
        bar = "#" * max(1, int(p["pct"] / 2))
        print(f"  {i:2}. {p['phrase']:<25} {bar:<20} {p['pct']}% ({p['count']} occurrences)")
    print("\nNEGATIVE SUMMARY")
    print("-" * 55)
    for i, p in enumerate(neg_summary, 1):
        bar = "#" * max(1, int(p["pct"] / 2))
        print(f"  {i:2}. {p['phrase']:<25} {bar:<20} {p['pct']}% ({p['count']} occurrences)")
    print("\n" + "=" * 55)


Processing 4638 reviews...



Phrase Extraction:   0%|          | 0/4638 [00:00<?, ?review/s]


Extraction complete in 38m 45s
Total phrases extracted: 5842
Phrases after SVM Lexicon filter: 2428

Predicting phrase sentiment...


Sentiment Prediction:   0%|          | 0/2428 [00:00<?, ?phrase/s]


Phrases accepted: 2393
Phrases rejected: 35 (confidence < 0.75)
Running semantic deduplication using SVM feature similarity weights...
Unique positive clusters: 414
Unique negative clusters: 283

Pipeline complete in 38m 45s

          AUTOMATED REVIEW SUMMARY

POSITIVE SUMMARY
-------------------------------------------------------
   1. pengiriman cepat          ##                   5.5% (255 occurrences)
   2. packaging aman            #                    2.7% (123 occurrences)
   3. barang bagus              #                    1.8% (82 occurrences)
   4. pesan cepat               #                    1.5% (68 occurrences)
   5. produk bagus              #                    1.4% (63 occurrences)
   6. proses pesan cepat        #                    1.4% (63 occurrences)
   7. produk sesuai             #                    1.2% (55 occurrences)
   8. respon cepat              #                    1.1% (49 occurrences)
   9. ukur sesuai               #                    1.0% (47 

## Phase 9 - Save All Outputs

In [15]:
phrase_summary = {
    "phrase_stats": phrase_stats,
    "positive"    : pos_summary or [],
    "negative"    : neg_summary or [],
}
phrase_summary_path = os.path.join(RESULT_DIR, "svm_phrase_summary.json")
with open(phrase_summary_path, "w", encoding="utf-8") as f:
    json.dump(phrase_summary, f, indent=2, ensure_ascii=False)
print(f"Phrase summary saved to {phrase_summary_path}")

print("\nAll outputs:")
for out_dir, label in [(MODEL_DIR, "model_weights"), (EVAL_DIR, "evaluation_metrics"), (RESULT_DIR, "results")]:
    print(f"\n  [{label}]")
    for fname in sorted(os.listdir(out_dir)):
        fpath = os.path.join(out_dir, fname)
        if os.path.isfile(fpath):
            size_kb = os.path.getsize(fpath) / 1024
            print(f"    {fname:<45} {size_kb:>8.1f} KB")

Phrase summary saved to /kaggle/working/results/svm_phrase_summary.json

All outputs:

  [model_weights]
    svm_model.joblib                                 310.1 KB
    tfidf_svm_vectorizer.joblib                       68.2 KB

  [evaluation_metrics]
    svm_all_metrics.json                               3.2 KB
    svm_confusion_matrix.png                          27.5 KB
    svm_roc_pr_curves.png                             74.7 KB
    svm_train_vs_val_metrics.png                      34.7 KB

  [results]
    svm_lexicon.json                                  53.7 KB
    svm_phrase_summary.json                            2.4 KB
